# Knapsack

In [2]:
def read_instance(path):
    with open(path, "r", encoding="utf-8") as f:
        n, W = map(int, f.readline().split())
        items = []

        for i in range(n):
            v, w = map(int, f.readline().split())
            items.append((v, w, i))

    return n, W, items

## Жадный алгоритм

In [13]:
def greedy_knapsack(n, W, items):
    sorted_items = sorted(items, key=lambda x: (x[0] / x[1], x[0]), reverse=True)
    total_value = 0
    total_weight = 0
    taken = [0] * n
    for v, w, i in sorted_items:
        if total_weight + w <= W:
            total_weight += w
            total_value += v
            taken[i] = 1
    return total_value, taken


In [14]:
tests = [
    "data/ks_30_0",
    "data/ks_50_0",
    "data/ks_200_0",
    "data/ks_400_0",
    "data/ks_1000_0",
    "data/ks_10000_0",
]

for test in tests:
    n, W, items = read_instance(test)
    value, taken = greedy_knapsack(n, W, items)
    print(test, value)


data/ks_30_0 90000
data/ks_50_0 141956
data/ks_200_0 100062
data/ks_400_0 3966813
data/ks_1000_0 109869
data/ks_10000_0 1099870


## Разные сортировки

In [15]:
def greedy_knapsack(n, W, items):
    def run_greedy(sorted_items):
        total_value = 0
        total_weight = 0
        taken = [0] * n
        for v, w, i in sorted_items:
            if total_weight + w <= W:
                total_weight += w
                total_value += v
                taken[i] = 1
        return total_value, taken
    return max(
        run_greedy(sorted(items, key=lambda x: (x[0] / x[1], x[0]), reverse=True)),
        run_greedy(sorted(items, key=lambda x: x[0], reverse=True)),
        run_greedy(sorted(items, key=lambda x: x[1])),
        key=lambda x: x[0])

In [16]:
tests = [
    "data/ks_30_0",
    "data/ks_50_0",
    "data/ks_200_0",
    "data/ks_400_0",
    "data/ks_1000_0",
    "data/ks_10000_0",
]

for test in tests:
    n, W, items = read_instance(test)
    value, taken = greedy_knapsack(n, W, items)
    print(test, value)


data/ks_30_0 99045
data/ks_50_0 142156
data/ks_200_0 100062
data/ks_400_0 3966825
data/ks_1000_0 109869
data/ks_10000_0 1099870


## Branch and Bound
Алгоритм рабоатет так: у нас есть дерево с ветками, где ветки -- предметы, бинарны варианты: брать или не брать. Посчитав на дробном рюкзаке, оптимум, мы понимаем, стоит ли продолжать эту ветку дерева или нет, потмоу что дробный рюкзак нам дает верхнюю границу (upper bound) для текущей ветки. Чтобы выжать из этой идеи максимум скорости и не упереться в лимит рекурсии, предметы предварительно сортируются по цене к вес и алгоритм делает быстрый жадный проход. Этот проход сразу дает нижнюю границу.

Дальше рекурсию заменяет обычный итеративный стек, работающий по принципу LIFO. В стек заталкивается сначала вариант не брать, а затем брать, в итоге pop() достает вариант брать первым, и алгоритм мгновенно проваливается вглубь по лучшим айтемам. Перед добавлением  нового состояния проверяется верхняя планка: если потенциал ветки с учетом дробления предметов ниже или равен нашему лучшему best, она срезается и не тратит память стека. У меня алгоритм отрабатывает за 7 секунд на все тесты и проходит все пороги.

In [ ]:
def bound(i, weight, value, items, W):
    result = value

    while i < len(items) and weight + items[i][1] <= W:
        weight += items[i][1]
        result += items[i][0]
        i += 1

    if i < len(items):
        result += (W - weight) * items[i][0] / items[i][1]

    return result


def search(items, W):
    best = 0
    curr_w = 0
    for item in items:
        v, w = item[0], item[1]
        if curr_w + w <= W:
            curr_w += w
            best += v
        else:
            break

    stack = [(0, 0, 0)]

    while stack:
        i, weight, value = stack.pop()

        if value > best:
            best = value

        if i == len(items):
            continue

        v, w = items[i][0], items[i][1]

        if bound(i + 1, weight, value, items, W) > best:
            stack.append((i + 1, weight, value))

        if weight + w <= W:
            if bound(i + 1, weight + w, value + v, items, W) > best:
                stack.append((i + 1, weight + w, value + v))

    return best


def branch_and_bound_knapsack(n, W, items):
    items = sorted(items, key=lambda x: x[0] / x[1], reverse=True)
    return search(items, W)

tests = [
    "data/ks_30_0",
    "data/ks_50_0",
    "data/ks_200_0",
    "data/ks_400_0",
    "data/ks_1000_0",
    "data/ks_10000_0",
]

for test in tests:
    n, W, items = read_instance(test)

    value = branch_and_bound_knapsack(n, W, items)

    print(test, value)

data/ks_30_0 99798
data/ks_50_0 142156
data/ks_200_0 100236
data/ks_400_0 3967180
data/ks_1000_0 109899
data/ks_10000_0 1099893
